# View dataset 4 (DINO crop)

10 random images per class from `04_denoise_clahe_dino`: dataset 2 cropped to the union of its Grounding DINO boxes, real pixels kept (no mask).

Each title shows the image id and **box**, the share of the dataset-2 image covered by the union box. A box near 100% means the crop barely changed the image. **FALLBACK** (red) means no box or a box under 5%, so the image is the dataset-2 image unchanged.

Set `COMPARE = True` to also see the same images in datasets 2 and 3.

Test images are shown for looking only. Never change the pipeline or pick a dataset because of what you see in test.

In [ ]:
SPLIT = "train"     # train, val or test
SEED = 0            # change to draw 10 other images per class
PER_CLASS = 10
COMPARE = False     # True: also show the same images in datasets 2 and 3

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data")) / "dataset"

CLASSES = ["bibimbap", "miso_soup", "mussels", "prime_rib", "edamame",
           "baby_back_ribs", "paella", "chicken_wings", "guacamole", "nachos"]

stats = pd.read_csv(DATA / "meta" / "dino_crop_stats.csv").set_index("relpath")
assert (DATA / "04_denoise_clahe_dino").is_dir(), "unzip 04_denoise_clahe_dino.zip into data/dataset/ first"
stats.groupby("split").agg(images=("fallback", "size"), fallback=("fallback", "sum"),
                           median_box=("box_area", "median")).round(2)

In [ ]:
# draw the image list once, so COMPARE shows the same images
rng = random.Random(SEED)
s = stats[stats.split == SPLIT]
picked = {c: rng.sample(sorted(r for r in s.index if r.split("/")[1] == c), PER_CLASS) for c in CLASSES}

## 10 images per class

In [ ]:
def show(dataset, rels, title):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
    for ax, rel in zip(axes.ravel(), rels):
        img = Image.open(DATA / dataset / rel)
        st = stats.loc[rel]
        fb = st.fallback == 1
        ax.imshow(img)
        ax.set_title(f"{Path(rel).stem}  box {st.box_area:.0%}" + ("  FALLBACK" if fb else ""),
                     fontsize=10, color="red" if fb else "black")
        ax.axis("off")
    fig.suptitle(title, fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()


for cls in CLASSES:
    show("04_denoise_clahe_dino", picked[cls], f"{cls}: 04_denoise_clahe_dino ({SPLIT})")
    if COMPARE:
        for ds in ["02_denoise_clahe", "03_denoise_clahe_gsam"]:
            if (DATA / ds).is_dir():
                show(ds, picked[cls], f"{cls}: {ds} ({SPLIT})")